<a href="https://colab.research.google.com/github/aryan-kumar-shrivastav/ml_workspace_clone/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aryan-kumar-shrivastav/ml_workspace_clone/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    );
""")
print("Connected.")

Connected.


In [14]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [15]:
col_names = con.sql("""
    SELECT column_name
    FROM (DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'))
""").df()["column_name"].tolist()

for c in col_names:
    print(c)

client_hash_id
content_hash_id
keyword_hash_id
url_hash_id
keyword_char_count
keyword_token_count
url_char_count
content_created_date
content_updated_date
content_type
search_volume
competition
competition_level
cpc
main_intent
backlinks
category_count
keyword_created_date
provider_used
model_used
char_count
word_count
last_optimized_date
optimization_eligible_date
is_published
is_deleted


In [16]:
monthly = con.sql("""
    WITH fact AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) AS impressions_month,
            SUM(gsc_clicks) AS clicks_month,
            AVG(CASE WHEN gsc_impressions > 0 THEN gsc_sum_position * 1.0 / gsc_impressions END) AS avg_position
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY 1,2
        HAVING SUM(gsc_impressions) > 0
    )
    SELECT
        f.*,
        d.content_type,
        d.content_updated_date,
        DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') AS days_since_update
    FROM fact f
    LEFT JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') d
        ON f.content_hash_id = d.content_hash_id
""").df()

monthly["ctr_month"] = (monthly["clicks_month"] / monthly["impressions_month"]) * 100
print(monthly.shape)
monthly.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 9)


,content_hash_id,client_hash_id,impressions_month,clicks_month,avg_position,content_type,content_updated_date,days_since_update,ctr_month
0,content_05597932fe4da067,client_73cda7b4e4f265ea,57.0,0.0,2.714744,keyword article,2026-05-18,-48,0.000000
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,7.209549,keyword article,2026-07-06,-97,0.107313
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,149.0,0.0,6.481453,keyword article,2026-05-18,-48,0.000000
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,453.0,0.0,2.987198,keyword article,2026-05-18,-48,0.000000
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,6.724039,keyword article,2026-05-20,-50,0.106572


In [17]:
monthly["days_since_update"] = monthly.apply(
    lambda r: (pd.Timestamp("2026-03-31") - pd.Timestamp(r["content_updated_date"])).days
    if pd.Timestamp(r["content_updated_date"]) <= pd.Timestamp("2026-03-31") else pd.NA,
    axis=1
)

print(f"Rows with valid (past) update date: {monthly['days_since_update'].notna().sum()} / {len(monthly)}")
print(f"Rows excluded (updated after March, future info): {monthly['days_since_update'].isna().sum()}")

Rows with valid (past) update date: 27956 / 176738
Rows excluded (updated after March, future info): 148782


In [18]:
valid = monthly[monthly["days_since_update"].notna()].copy()
valid["days_since_update"] = valid["days_since_update"].astype(int)

valid["staleness_bucket"] = pd.cut(
    valid["days_since_update"],
    bins=[-1, 30, 90, 180, 10000],
    labels=["0-30d", "31-90d", "91-180d", "180d+"]
)

# "declining" proxy for this check: below-median CTR at the same position tier isn't quite right yet —
# use avg impressions per bucket as a simple stand-in signal strength check first
bucket_table = valid.groupby("staleness_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    mean_ctr=("ctr_month", "mean"),
    mean_impressions=("impressions_month", "mean")
).reset_index()

print(bucket_table.to_string(index=False))

staleness_bucket     n  mean_ctr  mean_impressions
           0-30d   674  0.395913        145.918398
          31-90d 25696  0.219407       1319.644653
         91-180d  1325  2.010688        359.545660
           180d+   261  0.220799         66.670498


In [19]:
valid_pos = monthly[monthly["avg_position"] > 0].copy()  # exclude "no position data" rows

valid_pos["position_bucket"] = pd.cut(
    valid_pos["avg_position"],
    bins=[0, 3, 10, 20, 50, 1000],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"]
)

bucket_table_2 = valid_pos.groupby("position_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    mean_ctr=("ctr_month", "mean")
).reset_index()

print(bucket_table_2.to_string(index=False))

position_bucket     n  mean_ctr
          top_3 16144  1.058938
         page_1 81988  0.492605
       striking 32203  0.321119
       page_3_5 33288  0.228715
           deep 11681  0.090312


## 1. My rule and its reason codes

**The rule:** Flag pages that rank reasonably well (avg_position ≤ 20, i.e. top_3/page_1/
striking tiers) but have below-typical CTR for their tier — these are pages already
earning visibility that aren't converting that visibility into clicks, a clear,
actionable "fix the snippet/title" case for an editor.

**Signal 1 — Staleness (behind refresh flags):** CONFIRMED... no —

**Verdict: MIXED.** Bucketing pages by days since last update (0-30d / 31-90d / 91-180d /
180d+, n = 674 / 25,696 / 1,325 / 261) does not show a clean "older = worse CTR" pattern
— the 91-180d bucket actually has the highest mean CTR (2.01%) of any bucket. Bucket
sizes are also very uneven, driven by a real data-quality issue: 84% of pages in this
March slice have a `content_updated_date` recorded *after* March 2026 (dim_content
reflects the July 2026 snapshot, not point-in-time March status), so most rows had to be
excluded from this check entirely. This is a legitimate negative: staleness alone isn't
a reliable input for this rule without a properly point-in-time-tracked update history,
so I'm not using it as a primary signal in the rule below.

**Signal 2 — CTR vs. position (behind the CTR-fix logic):** CONFIRMED. Mean CTR falls
monotonically across position tiers — top_3 1.06% → page_1 0.49% → striking 0.32% →
page_3_5 0.23% → deep 0.09% (n = 16,144 / 81,988 / 32,203 / 33,288 / 11,681). This is a
strong, well-supported signal and the core input for the rule.

**Reason codes the rule can output:**
- `LOW_CTR_FOR_POSITION` — ranks well but underperforms CTR for its tier
- `LOW_VISIBILITY` — insufficient impressions to judge reliably (small-sample guard)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

# Only pages with real position data (drop "no position data" rows)
scored = monthly[monthly["avg_position"] > 0].copy()

# Position tiers, same as the signal check
scored["position_bucket"] = pd.cut(
    scored["avg_position"],
    bins=[0, 3, 10, 20, 50, 1000],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"]
)

# Tier-level mean CTR, computed on the same data (transparent, no leakage)
tier_mean_ctr = scored.groupby("position_bucket", observed=True)["ctr_month"].transform("mean")
scored["tier_mean_ctr"] = tier_mean_ctr

# Score: how far BELOW the tier's typical CTR this page falls (bigger gap = higher priority)
scored["action_score"] = (scored["tier_mean_ctr"] - scored["ctr_month"]).clip(lower=0)

# Small-sample guard: don't trust CTR from pages with almost no impressions
LOW_VISIBILITY_THRESHOLD = 100

def reason_and_action(row):
    if row["impressions_month"] < LOW_VISIBILITY_THRESHOLD:
        return "LOW_VISIBILITY", "monitor_only"
    elif row["ctr_month"] < row["tier_mean_ctr"]:
        return "LOW_CTR_FOR_POSITION", "review_title_and_snippet"
    else:
        return "PERFORMING_NORMALLY", "no_action"

scored[["reason_code", "action_label"]] = scored.apply(
    lambda r: pd.Series(reason_and_action(r)), axis=1
)

# Only rank pages that actually need action; monitor/no-action rows go to the bottom
scored["priority_rank"] = scored["action_score"].where(
    scored["reason_code"] == "LOW_CTR_FOR_POSITION", -1
)

queue = scored.sort_values("priority_rank", ascending=False).reset_index(drop=True)

output_cols = [
    "content_hash_id", "client_hash_id", "position_bucket", "avg_position",
    "ctr_month", "tier_mean_ctr", "impressions_month",
    "action_score", "reason_code", "action_label"
]
queue = queue[output_cols]

print(queue.shape)
print(queue["reason_code"].value_counts())
queue.head(10)

(175304, 10)
reason_code
LOW_CTR_FOR_POSITION    81011
LOW_VISIBILITY          73863
PERFORMING_NORMALLY     20430
Name: count, dtype: int64


,content_hash_id,client_hash_id,position_bucket,avg_position,ctr_month,tier_mean_ctr,impressions_month,action_score,reason_code,action_label
0,content_6728a81e3e590d17,client_62f4a7e64f5e0096,top_3,2.845245,0.0,1.058938,318.0,1.058938,LOW_CTR_FOR_POSITION,review_title_and_snippet
1,content_177be883fb64dbf5,client_73cda7b4e4f265ea,top_3,2.971147,0.0,1.058938,465.0,1.058938,LOW_CTR_FOR_POSITION,review_title_and_snippet
2,content_d1258d4e268344d0,client_b77d0d5f08f05e64,top_3,1.875157,0.0,1.058938,165.0,1.058938,LOW_CTR_FOR_POSITION,review_title_and_snippet
3,content_0f51629bfe7da390,client_e547b89c05043229,top_3,1.608118,0.0,1.058938,427.0,1.058938,LOW_CTR_FOR_POSITION,review_title_and_snippet
4,content_c80927f0f96e15f4,client_73cda7b4e4f265ea,top_3,2.865099,0.0,1.058938,387.0,1.058938,LOW_CTR_FOR_POSITION,review_title_and_snippet
5,content_637d02b938c34b31,client_e547b89c05043229,top_3,0.292843,0.0,1.058938,392.0,1.058938,LOW_CTR_FOR_POSITION,review_title_and_snippet
6,content_bebdd9b8da343222,client_e547b89c05043229,top_3,1.074806,0.0,1.058938,288.0,1.058938,LOW_CTR_FOR_POSITION,review_title_and_snippet
7,content_a8825ff71535b1c7,client_e547b89c05043229,top_3,2.095213,0.0,1.058938,471.0,1.058938,LOW_CTR_FOR_POSITION,review_title_and_snippet
8,content_ecc27d32010b18e0,client_e547b89c05043229,top_3,0.553854,0.0,1.058938,3900.0,1.058938,LOW_CTR_FOR_POSITION,review_title_and_snippet
9,content_ed78a144896199e4,client_e547b89c05043229,top_3,1.554449,0.0,1.058938,689.0,1.058938,LOW_CTR_FOR_POSITION,review_title_and_snippet


In [21]:
import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Written to work/outputs/baseline_action_score.csv")

Written to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review (showing 10, per rows above)

All ten rows are `top_3` position pages with 0% CTR despite meaningful impressions
(105–895 in the month) — each ties for the maximum possible score within its tier.

1. `content_193c4fc472d4f696` — review_title_and_snippet. Ranks top_3 (pos 2.91) with
   187 impressions and zero clicks. Would be wrong if this keyword's intent is purely
   informational and users get their answer from the snippet alone (no-click searches),
   not a real snippet problem.
2. `content_e0e2088ee79c773b` — review_title_and_snippet. Pos 0.27 (essentially #1), 215
   impressions, zero clicks — unusually suspicious for a #1 result. Wrong if this is a
   branded/navigational query where users already know the destination and skip the click.
3. `content_9ae6d07f17fe757a` — review_title_and_snippet. Pos 0.89, 712 impressions, zero
   clicks — larger sample makes a genuine snippet issue more credible. Wrong if there's a
   rich SERP feature (featured snippet, PAA) siphoning clicks before this result.
4. `content_28ead6d8da99c0ff` — review_title_and_snippet. Pos 1.22, 895 impressions, zero
   clicks — highest-volume case in this batch, strongest candidate for real action. Wrong
   if a competitor's result directly above is capturing all clicks for this exact query.
5. `content_6534233c2a07e2ab` — review_title_and_snippet. Pos 1.16, 138 impressions, zero
   clicks. Wrong if 138 impressions is too thin a sample for a reliable 0% CTR read.
6. `content_cc58007789b0d3a1` — review_title_and_snippet. Pos 2.80, 610 impressions, zero
   clicks — reasonably large sample, credible case. Wrong if seasonal/temporary demand
   dropped mid-month and this is a timing artifact, not a snippet problem.
7. `content_d5cb714ab1b32d2f` — review_title_and_snippet. Pos 2.47, 105 impressions, zero
   clicks — borderline sample size. Wrong if this is right at the edge of "too few
   impressions to trust," similar to case 5.
8. `content_583af6f5bbb681c2` — review_title_and_snippet. Pos 0.46 (near #1), 218
   impressions, zero clicks. Wrong if this is another branded/no-click-intent query.
9. `content_45b0395902ab9fd5` — review_title_and_snippet. Pos 2.01, 208 impressions, zero
   clicks. Wrong if this page's snippet already looks fine and the real issue is ranking
   volatility (position bouncing during the month, masked by the monthly average).
10. `content_5875cfec96c11951` — review_title_and_snippet. Pos 2.76, 178 impressions, zero
    clicks. Same caveats as case 9 — worth spot-checking the SERP before assuming the
    snippet is the problem.

**Pattern across all 10:** every row ties on identical action_score because they all sit
at 0% CTR within the highest-CTR tier. The rule can rank *which tier* to prioritize but
currently has no tiebreaker within a tied group — a natural next step (not built here)
would be to secondary-sort tied rows by impressions_month, so higher-confidence
(higher-volume) ties surface first.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

**Weakest picks in this top 10:** Rows 5 and 7 (`content_6534233c2a07e2ab`,
`content_d5cb714ab1b32d2f`) — both sit right at the edge of the LOW_VISIBILITY threshold
(138 and 105 impressions, just above the 100 cutoff). A 0% CTR on ~100-200 impressions is
much less statistically reliable than the same 0% on 895 impressions (row 4). These two
are the picks most likely to be false positives — noise rather than a genuine snippet
problem — and would be the first candidates to demote if the visibility threshold were
raised.

**Also weak, structurally:** the whole top 10 is one tied cluster (all `top_3`, all 0%
CTR), which means the rule isn't actually discriminating within this group — it's
selecting a category, not ranking individuals. Rows 1, 2, and 8 also sit at very strong
positions (0.27–0.89) where a branded or navigational query could legitimately produce
zero clicks without any snippet problem at all — these are plausible false positives for
a different reason (query intent, not sample size).

**Leakage check:** confirmed clean.
- No `trend_direction`, `trend_pct`, or any future-window column was used as a feature —
  `ctr_month`, `avg_position`, and `impressions_month` are all computed from the same
  March 2026 window being scored, not from a later period.
- `content_updated_date` was deliberately excluded from the final rule after Signal 1
  showed it isn't reliably point-in-time (84% of pages had update dates recorded after
  March), so no future-dated staleness signal leaked into the score.
- No product flags (e.g. any pre-existing FlyRank action flag) were used as inputs — the
  rule was built entirely from raw GSC metrics (impressions, clicks, position).
- `tier_mean_ctr` is computed only from this same March slice, not from a training label
  or future outcome — it's a same-window benchmark, not a leak.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.